# Notebook 04: Command Metrics

Compute and inspect rolling command metrics for a selected game:
- Rolling horizontal and vertical spread
- Rolling miss distance from location hub
- Cluster drift (early / mid / late)

All computation is done via service layer calls.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.services.pitcher_service import (
    get_pitcher_dataset,
    get_available_games,
    get_available_pitch_types,
    fit_pitch_type_clusters,
    compute_game_command_metrics,
    compute_cluster_drift,
)
from src.features.clustering import assign_clusters_to_game
from src.data.fetch import resolve_pitcher_id
from src import config

pd.set_option('display.float_format', '{:.3f}'.format)
%matplotlib inline

In [ ]:
# ---- Parameters ----
PITCHER    = "Gerrit Cole"
START_DATE = "2024-04-01"
END_DATE   = "2024-09-30"
PITCH_TYPE = "FF"
WINDOW     = 7
K          = 4
GAME_PK    = None   # Set to an integer game_pk, or None to use the first available
# --------------------

df = get_pitcher_dataset(PITCHER, START_DATE, END_DATE)
pitcher_id = resolve_pitcher_id(PITCHER)

games = get_available_games(df)
if GAME_PK is None:
    GAME_PK = games[0].game_pk
    print(f"Using first available game: {games[0]}")
else:
    print(f"Using game_pk: {GAME_PK}")

## Fit baseline clusters

In [ ]:
cluster_model = fit_pitch_type_clusters(df, pitcher_id, PITCH_TYPE, k=K)

if cluster_model is None:
    print(f"Cannot fit clusters for {PITCH_TYPE} — try another pitch type or wider date range.")
else:
    print(f"Cluster model: k={cluster_model.k}, n_fit={cluster_model.n_pitches_fit}")

## Assign clusters to game

In [ ]:
if cluster_model:
    game_df = assign_clusters_to_game(df, GAME_PK, PITCH_TYPE, cluster_model)
    print(f"Game pitches (type={PITCH_TYPE}): {len(game_df)}")
    print(f"Cluster counts: {game_df['cluster_label'].value_counts().to_dict()}")
    game_df[['game_pitch_index', 'plate_x', 'plate_z', 'cluster_label', 'miss_distance', 'game_bucket']].head(10)

## Rolling metrics

In [ ]:
if cluster_model:
    metrics = compute_game_command_metrics(
        df=df,
        game_pk=GAME_PK,
        pitch_type=PITCH_TYPE,
        window=WINDOW,
        cluster_model=cluster_model,
    )
    
    spreads = metrics['rolling_spreads']
    miss = metrics['rolling_miss']
    
    print(f"Rolling spreads: {len(spreads)} rows")
    print(f"Rolling miss: {len(miss)} rows")
    spreads.head()

## Cluster drift (early / mid / late)

In [ ]:
if cluster_model:
    drift = compute_cluster_drift(
        df=df,
        game_pk=GAME_PK,
        pitch_type=PITCH_TYPE,
        cluster_model=cluster_model,
    )
    
    print(f"Overall drift magnitude: {drift.overall_drift_magnitude:.3f} ft")
    print(f"Total pitches: {drift.n_pitches}")
    print()
    
    for s in drift.cluster_summaries:
        print(f"Cluster {s.cluster_id} (baseline: x={s.baseline_center[0]:.2f}, z={s.baseline_center[1]:.2f}):")
        for bucket in ['early', 'mid', 'late']:
            mean = getattr(s, f'{bucket}_mean')
            mag = getattr(s, f'{bucket}_drift_magnitude')
            n = getattr(s, f'{bucket}_n')
            if mean is not None:
                print(f"  {bucket}: mean=({mean[0]:.2f}, {mean[1]:.2f}), drift={mag:.3f} ft, n={n}")
            else:
                print(f"  {bucket}: insufficient data (n={n})")